In [ ]:
import requests, json, pandas as pd, time, re, ast, sys
from requests.exceptions import ReadTimeout, ConnectionError, HTTPError, RequestException

BASE_URL = "http://127.0.0.1:8080/v1"
MODEL    = "/home/dervic/txagent-t1-8b.Q4_K_M.gguf"

ses = requests.Session()
ses.headers.update({"Authorization": "Bearer sk-local", "Content-Type": "application/json"})

SYSTEM_PROMPT = """
You answer medical evidence questions in a permissive (loose) way.

Think step-by-step internally, but DO NOT reveal full chain-of-thought.
Return JSON only, no extra text.

Schema:
{
  "decision_binary": 0 or 1,
  "reasoning_short": "short reason (1-2 sentences OR up to 3 bullets)"
}

LOOSE decision rule:
- Output 1 if there is ANY evidence or widely accepted clinical use for the drug for:
  (a) at least one disease/condition in the trajectory, OR
  (b) symptom management plausibly relevant to the trajectory (e.g., nausea, agitation, insomnia, pain),
  even if it does not treat the infections or the full combination.
- Output 0 only if there is no reasonable match to any component condition and no plausible symptom-management use,
  or if it is clearly contraindicated/unsafe for the situation.

Keep reasoning_short brief:
- Mention what it matches (condition or symptom) and the evidence type (guideline/RCT/observational/case/practice/mechanistic),
  or say "no match".
No markdown fences.

Few-shot examples (format you must follow):

Example 1 input:
Trajectory diseases: mental and behavioural disorders due to use of opioids; mental and behavioural disorders due to use of sedatives or hypnotics; mental and behavioural disorders due to use of multiple drug use and use of psychoactive substances
Predicted drug: Quetiapine

Example 1 output:
{"decision_binary":1,"reasoning_short":"Commonly used for agitation/psychosis in substance-related presentations; indirect clinical support (observational/accepted practice). Not specific to the full combo."}

Example 2 input:
Trajectory diseases: hypothyroidism; iron deficiency anaemia; anaemias; thyroiditis
Predicted drug: Levothyroxine

Example 2 output:
{"decision_binary":1,"reasoning_short":"Standard of care for hypothyroidism (guidelines). Treats thyroid component; anemia management is separate."}
""".strip()

def clean_diseases_str(diseases):
    if diseases is None:
        return ""
    if isinstance(diseases, list):
        return ", ".join([str(x).strip() for x in diseases if str(x).strip()])

    s = str(diseases).strip()
    if not s:
        return ""

    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return ", ".join([str(x).strip() for x in parsed if str(x).strip()])
    except Exception:
        pass

    s = s.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_json_object(raw_text):
    raw = (raw_text or "").strip()

    # direct parse
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # strip code fences
    raw2 = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.IGNORECASE | re.DOTALL).strip()
    try:
        obj = json.loads(raw2)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # find first {...}
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass

    return None

def coerce_output(obj):
    out = {"decision_binary": 0, "reasoning_short": "No reasonable match to any condition or symptom-management use."}

    try:
        db = int(obj.get("decision_binary", 0))
    except Exception:
        db = 0
    out["decision_binary"] = 1 if db == 1 else 0

    rs = obj.get("reasoning_short", out["reasoning_short"])
    rs = "" if rs is None else str(rs).strip()
    rs = re.sub(r"\s+", " ", rs).strip()
    if not rs:
        rs = out["reasoning_short"]
    if len(rs) > 320:
        rs = rs[:317] + "..."
    out["reasoning_short"] = rs

    return out

def ask(diseases, drug, trajectory, icd_list):
    diseases_str = clean_diseases_str(diseases)
    drug = "" if drug is None else str(drug).strip()

    if not diseases_str or not drug:
        return {"decision_binary": 0, "reasoning_short": "Missing diseases or drug."}

    user_prompt = (
        f"You have trajectory consisting of the following diseases: {diseases_str}\n"
        f"A predicted drug for this trajectory is: {drug}.\n\n"
        f"Is there any evidence that using {drug} for this combination of these conditions is reasonable?\n\n"
        "Return JSON only with keys decision_binary and reasoning_short."
    )

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0,
        "max_tokens": 220
    }

    backoff = 2
    for attempt in range(4):
        try:
            r = ses.post(f"{BASE_URL}/chat/completions", json=payload, timeout=180)
            r.raise_for_status()
            raw = (r.json()["choices"][0]["message"]["content"] or "").strip()

            obj = extract_json_object(raw)
            if obj is None:
                # parse failed, quick retry
                print(f"[warn] parse failed (attempt {attempt+1})", flush=True)
                time.sleep(1)
                continue

            return coerce_output(obj)

        except ReadTimeout:
            print(f"[warn] ReadTimeout (attempt {attempt+1}) sleeping {backoff}s", flush=True)
            time.sleep(backoff)
            backoff = min(backoff * 2, 30)
            continue

        except ConnectionError as e:
            print(f"[warn] ConnectionError (attempt {attempt+1}) sleeping {backoff}s: {str(e)[:120]}", flush=True)
            time.sleep(backoff)
            backoff = min(backoff * 2, 30)
            continue

        except HTTPError as e:
            print(f"[error] HTTPError: {str(e)[:160]}", flush=True)
            return {"decision_binary": 0, "reasoning_short": f"HTTP error: {str(e)[:120]}"}

        except RequestException as e:
            print(f"[error] RequestException: {str(e)[:160]}", flush=True)
            return {"decision_binary": 0, "reasoning_short": f"Request error: {str(e)[:120]}"}

        except Exception as e:
            print(f"[error] Unexpected: {str(e)[:160]}", flush=True)
            return {"decision_binary": 0, "reasoning_short": f"Unexpected error: {str(e)[:120]}"}

    return {"decision_binary": 0, "reasoning_short": "Timeout/connection failure after retries."}

# -----------------------------
# Quick connectivity check (fail fast)
# -----------------------------
try:
    ping = ses.get(f"{BASE_URL}/models", timeout=5)
    print(f"[startup] GET /models status={ping.status_code}", flush=True)
except Exception as e:
    print(f"[startup] model server not responding at {BASE_URL}: {e}", flush=True)

# -----------------------------
# Run your sheet
# -----------------------------
INPUT = "NewFilesV2/random_male_trj_mol_drug_v2026_official.csv"
OUT   = "NewFilesV2/random_male_trj_mol_drug_v2026_officialTaxAgentReasoning.xlsx"

df = pd.read_csv(INPUT)

if "Result" not in df.columns:
    df["Result"] = ""
if "reasoning_short" not in df.columns:
    df["reasoning_short"] = ""

cache = {}
SAVE_EVERY = 50

print(f"[startup] rows={len(df)} saving_to={OUT}", flush=True)

for i, row in df.iterrows():
    diseases   = "" if pd.isna(row["Disease Consensus Name List"]) else str(row["Disease Consensus Name List"]).strip()
    drug       = "" if pd.isna(row["Drug"]) else str(row["Drug"]).strip()
    trajectory = row["Trajectory"]

    # PRINT BEFORE LLM CALL (so you always see progress)
    print(f"[{i}] querying drug={drug}", flush=True)

    if not diseases:
        out = {"decision_binary": 0, "reasoning_short": "Missing diseases."}
    else:
        key = (diseases, drug)
        if key not in cache:
            cache[key] = ask(diseases, drug, trajectory, [])
        out = cache[key]

    df.at[i, "Result"] = str(int(out["decision_binary"]))
    df.at[i, "reasoning_short"] = out["reasoning_short"]

    print(f"[{i}] evidence: {df.at[i,'Result']} | {df.at[i,'reasoning_short']}", flush=True)

    if SAVE_EVERY and (i > 0) and (i % SAVE_EVERY == 0):
        df.to_excel(OUT, index=False)
        print(f"[checkpoint] saved -> {OUT}", flush=True)

df.to_excel(OUT, index=False)
print("Done ->", OUT, flush=True)
